# 14a — Model Evaluation: Metrics

Corresponds to: Note 14 — Model Evaluation: Metrics

**Libraries introduced in this notebook:** No new libraries — `torch` and `torch.nn` introduced in Note 09.

**What this notebook demonstrates:**
1. Confusion matrix — computed from scratch, verified against Note 14 Part 2 worked example
2. Precision, recall, F1 — computed from the confusion matrix
3. ROC curve and AUC — computed by varying the threshold
4. Precision-recall curve and average precision
5. Full evaluation pipeline — training a classifier and evaluating with all metrics

In [ ]:
import torch
import torch.nn as nn
import numpy as np

print(f'PyTorch version: {torch.__version__}')

## Part 1: Confusion Matrix

We implement the confusion matrix from scratch and verify it against the Note 14 Part 2 worked example:
100 samples, 40 positive, 60 negative — TP=28, FN=12, FP=8, TN=52.

In [ ]:
def confusion_matrix(y_true, y_pred):
    """Compute TP, FP, FN, TN for binary classification."""
    tp = ((y_pred == 1) & (y_true == 1)).sum().item()
    fp = ((y_pred == 1) & (y_true == 0)).sum().item()
    fn = ((y_pred == 0) & (y_true == 1)).sum().item()
    tn = ((y_pred == 0) & (y_true == 0)).sum().item()
    return tp, fp, fn, tn

# Reproduce Note 14 Part 2 worked example
# 40 positives: 28 TP, 12 FN
# 60 negatives: 8 FP, 52 TN
y_true = torch.tensor([1]*28 + [1]*12 + [0]*8  + [0]*52)
y_pred = torch.tensor([1]*28 + [0]*12 + [1]*8  + [0]*52)

tp, fp, fn, tn = confusion_matrix(y_true, y_pred)
print('Confusion matrix (Note 14 Part 2 example):')
print(f'  TP = {tp}  (expected 28)')
print(f'  FP = {fp}  (expected  8)')
print(f'  FN = {fn}  (expected 12)')
print(f'  TN = {tn}  (expected 52)')
print(f'  Accuracy = {(tp + tn) / (tp + fp + fn + tn):.3f}  (expected 0.800)')

## Part 2: Precision, Recall, F1

All three metrics computed from the confusion matrix. Verified against Note 14 Parts 3 and 4.

In [ ]:
def precision_recall_f1(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

precision, recall, f1 = precision_recall_f1(tp, fp, fn)
print('Metrics from Note 14 Part 2 example:')
print(f'  Precision = {precision:.3f}  (expected 0.778)')
print(f'  Recall    = {recall:.3f}  (expected 0.700)')
print(f'  F1        = {f1:.3f}  (expected 0.737)')
print()

# Demonstrate harmonic mean vs arithmetic mean
p_extreme, r_extreme = 1.0, 0.01
arith = (p_extreme + r_extreme) / 2
harm  = 2 * p_extreme * r_extreme / (p_extreme + r_extreme)
print('Harmonic vs arithmetic mean when P=1.0, R=0.01:')
print(f'  Arithmetic mean = {arith:.3f}  (misleadingly high)')
print(f'  Harmonic mean   = {harm:.3f}  (correctly low — recall is terrible)')

## Part 3: ROC Curve and AUC

The ROC curve is built by varying the classification threshold from 1 to 0 and computing
TPR and FPR at each threshold. AUC is the area under this curve.

We use simulated classifier scores — positives tend to score high, negatives tend to score low.

In [ ]:
rng = np.random.default_rng(42)
n_pos, n_neg = 100, 200
scores_pos = rng.beta(5, 2, n_pos)
scores_neg = rng.beta(2, 5, n_neg)
scores = np.concatenate([scores_pos, scores_neg])
labels = np.array([1] * n_pos + [0] * n_neg)

def roc_curve(scores, labels, n_thresholds=200):
    thresholds = np.linspace(0, 1, n_thresholds)
    tprs, fprs = [], []
    for t in thresholds:
        preds = (scores >= t).astype(int)
        tp = ((preds == 1) & (labels == 1)).sum()
        fp = ((preds == 1) & (labels == 0)).sum()
        fn = ((preds == 0) & (labels == 1)).sum()
        tn = ((preds == 0) & (labels == 0)).sum()
        tprs.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        fprs.append(fp / (fp + tn) if (fp + tn) > 0 else 0)
    return np.array(fprs), np.array(tprs)

fprs, tprs = roc_curve(scores, labels)
auc = -np.trapezoid(tprs, fprs)

print(f'AUC-ROC = {auc:.3f}')
print(f'Interpretation: a randomly chosen positive scores higher than a randomly chosen')
print(f'negative {auc*100:.1f}% of the time.')
print()
print('TPR and FPR at selected thresholds:')
print(f"{'Threshold':>10}  {'TPR':>8}  {'FPR':>8}")
print("-" * 32)
thresholds = np.linspace(0, 1, 200)
for i in [190, 160, 130, 100, 70, 40, 10]:
    print(f"{thresholds[i]:>10.2f}  {tprs[i]:>8.3f}  {fprs[i]:>8.3f}")

## Part 4: Precision-Recall Curve and Average Precision

The precision-recall curve is preferred over ROC when the positive class is rare.
We compute it on the same scores and compare the two curves.

In [ ]:
def pr_curve(scores, labels, n_thresholds=200):
    thresholds = np.linspace(0, 1, n_thresholds)
    precisions, recalls = [], []
    for t in thresholds:
        preds = (scores >= t).astype(int)
        tp = ((preds == 1) & (labels == 1)).sum()
        fp = ((preds == 1) & (labels == 0)).sum()
        fn = ((preds == 0) & (labels == 1)).sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        precisions.append(prec)
        recalls.append(rec)
    return np.array(recalls), np.array(precisions)

recalls_pr, precisions_pr = pr_curve(scores, labels)
sort_idx = np.argsort(recalls_pr)
ap = -np.trapezoid(precisions_pr[sort_idx], recalls_pr[sort_idx])
baseline = n_pos / (n_pos + n_neg)

print(f'Average Precision (AUC-PR) = {ap:.3f}')
print(f'Random classifier baseline = {baseline:.3f}  (fraction of positives in dataset)')
print()
print('Precision and Recall at selected thresholds:')
print(f"{'Threshold':>10}  {'Precision':>10}  {'Recall':>8}")
print("-" * 34)
thresholds = np.linspace(0, 1, 200)
for i in [190, 160, 130, 100, 70, 40, 10]:
    print(f"{thresholds[i]:>10.2f}  {precisions_pr[i]:>10.3f}  {recalls_pr[i]:>8.3f}")

## Part 5: Full Evaluation Pipeline

We train a binary classifier on a synthetic imbalanced dataset (10% positive)
and evaluate it with all metrics from the note.

In [ ]:
torch.manual_seed(0)
rng2 = np.random.default_rng(1)

# Imbalanced dataset: 10% positive
n_total = 500
n_pos_data = 50
X0 = rng2.normal([-1, -1], 1.0, (n_total - n_pos_data, 2))
X1 = rng2.normal([ 2,  2], 1.0, (n_pos_data, 2))
X_np = np.vstack([X0, X1]).astype(np.float32)
y_np = np.array([0]*(n_total - n_pos_data) + [1]*n_pos_data, dtype=np.float32).reshape(-1,1)
idx  = rng2.permutation(n_total)
X_all = torch.tensor(X_np[idx])
y_all = torch.tensor(y_np[idx])

split = 400
X_tr, y_tr = X_all[:split], y_all[:split]
X_te, y_te = X_all[split:], y_all[split:]

# Train
model = nn.Sequential(nn.Linear(2,16), nn.ReLU(), nn.Linear(16,1), nn.Sigmoid())
opt   = torch.optim.Adam(model.parameters(), lr=0.01)
crit  = nn.BCELoss()
model.train()
for _ in range(200):
    opt.zero_grad()
    crit(model(X_tr), y_tr).backward()
    opt.step()

# Evaluate
model.eval()
with torch.no_grad():
    scores_te = model(X_te).squeeze().numpy()
y_te_np = y_te.squeeze().numpy().astype(int)
preds_te = (scores_te >= 0.5).astype(int)

tp = int(((preds_te == 1) & (y_te_np == 1)).sum())
fp = int(((preds_te == 1) & (y_te_np == 0)).sum())
fn = int(((preds_te == 0) & (y_te_np == 1)).sum())
tn = int(((preds_te == 0) & (y_te_np == 0)).sum())

acc  = (tp + tn) / (tp + fp + fn + tn)
prec = tp / (tp + fp) if (tp + fp) > 0 else 0
rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

fprs_te, tprs_te = roc_curve(scores_te, y_te_np)
auc_te = -np.trapezoid(tprs_te, fprs_te)

print(f'Test set: {len(y_te_np)} samples, {y_te_np.sum()} positive ({y_te_np.mean()*100:.0f}%)')
print()
print(f'Confusion matrix:  TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print()
print(f'Accuracy  = {acc:.3f}')
print(f'Precision = {prec:.3f}')
print(f'Recall    = {rec:.3f}')
print(f'F1        = {f1:.3f}')
print(f'AUC-ROC   = {auc_te:.3f}')
print()
print('Note: on this imbalanced dataset, accuracy is high but may not reflect')
print('true performance on the positive class — check precision and recall.')

## Summary

This notebook demonstrated:

1. **Confusion matrix** — TP, FP, FN, TN computed from scratch. Verified against Note 14 Part 2: TP=28, FP=8, FN=12, TN=52, accuracy=80%.
2. **Precision, recall, F1** — all derived from the confusion matrix. Harmonic mean demonstrated to correctly penalise extreme imbalance between precision and recall that arithmetic mean would mask.
3. **ROC curve and AUC** — built by varying threshold from 1 to 0. AUC interpreted as the probability that a random positive scores higher than a random negative.
4. **Precision-recall curve** — does not involve TN, so is not diluted by the majority class. Average precision is the AUC of this curve. Random baseline equals the fraction of positives in the dataset.
5. **Full pipeline** — classifier trained on an imbalanced dataset (10% positive). All metrics computed and compared — accuracy is high but precision and recall reveal the true picture.